# Lab 04b: Interfaces, traits and checking the Ecosystem


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "docs", "src", "lecture_04"))
Pkg.develop(path=@__DIR__)

using Revise
using Ecosystem
using Random

  Activating project at `~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/docs/src/lecture_04`
   Resolving package versions...
     Project No packages added to or removed from `~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/docs/src/lecture_04/Project.toml`
    Manifest No packages added to or removed from `~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/docs/src/lecture_04/Manifest.toml`


## Part I: The World as a collection

So far, every function digs into `w.agents |> values |> collect`. The `World` is a collection of agents, so let it behave like one.

<div class="alert alert-block alert-success">
<b>Exercise:</b>
</div>
Implement the <a href="https://docs.julialang.org/en/v1/manual/interfaces/#man-interface-iteration">iteration interface</a> for `World`, iterating over its agents:

- `Base.iterate`
- `Base.length`
- `Base.eltype`

<b>Hints:</b> The agents are stored in a `Dict`, which is already iterable. Can you forward iteration to `values(w.agents)`? Define `eltype` on the type: `Base.eltype(::Type{World{A}}) where A`.


In [2]:
methods(Base.iterate, (World,))

# 1 method for generic function "iterate" from Base:
 [1] iterate(w::World, state...)
     @ ~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:33

In [3]:
length(World([Grass(1), Sheep(2)]))

2

In [4]:
eltype(World([Grass(1), Sheep(2)]))

Agent

Check what you get for free once `World` is iterable:

In [5]:
w = World([Grass(1), Sheep(2), Wolf(3)])
for a in w
    println(a)
end

collect(w)
count(a -> a isa Animal{Sheep}, w)
sum(agent_count, w)
filter(a -> eats(Wolf(4), a), collect(w))

🐺♀ #3 E=10.0 ΔE=8.0 pr=0.1 pf=0.2
🐑♂ #2 E=4.0 ΔE=0.2 pr=0.8 pf=0.6
🌿  #1 30% grown


1-element Vector{Agent}:
 🐑♂ #2 E=4.0 ΔE=0.2 pr=0.8 pf=0.6

<div class="alert alert-block alert-info">
<b>Discussion:</b> 

1. What happens with `collect(w)` if you do not define `length`? Which trait is responsible? 

(Works, but SizeUnknown)

2. Is it safe to iterate over `World` while `world_step!` adds and removes agents?

(Not safe)

- Rewrite `find_food` using the new interface. 

</div>

In [6]:
# `find_food` is implemented inside Ecosystem, where it uses `collect(w)`.
find_food(Sheep(1), World([Grass(2, 3, 3)]))

🌿  #2 100% grown

## Part II: Who eats whom? A `Diet` trait

The current rules are written species by species. Adding species makes that table grow quickly. Instead, describe what matters: herbivores eat plants and carnivores eat herbivores.

<div class="alert alert-block alert-success">
<b>Exercise:</b>
</div>

1. Define a trait `Diet` with values `Herbivore` and `Carnivore`.
2. Assign a `Diet` to `Sheep` and `Wolf`.
3. Replace the species-specific methods of `eats` and `eat!` with methods dispatching on the trait: a herbivore eats a plant with `size > 0`; a carnivore eats an animal whose species is a herbivore.
4. Check that the simulation behaves as before.

<b>Hints:</b> 

- Use `Diet(::Type{Sheep}) = Herbivore()`
- dispatch via `eats(a::Animal{A}, b::Agent) where A = eats(Diet(A), b)`. 
- Keep a fallback for anything that is not food.


In [7]:
# Do not redefine these in the notebook. They belong to Ecosystem.
(Ecosystem.Diet, Ecosystem.Herbivore, Ecosystem.Carnivore)

(Diet, Herbivore, Carnivore)

In [8]:
(Ecosystem.Diet(Sheep), Ecosystem.Diet(Wolf))

(Herbivore(), Carnivore())

In [9]:
# These trait-dispatch methods are implemented in Ecosystem.
(eats(Sheep(1), Grass(2, 3, 3)), eats(Wolf(1), Sheep(2)))

(true, true)

In [10]:
# `eat!` is also implemented in Ecosystem and dispatches through Ecosystem.Diet.
methods(Ecosystem.eat!)

# 4 methods for generic function "eat!" from Ecosystem:
 [1] eat!(a::Animal{A}, b::Agent, w::World) where A
     @ ~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:95
 [2] eat!(::Herbivore, a::Animal, p::Plant, ::World)
     @ ~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:96
 [3] eat!(::Carnivore, a::Animal, b::Animal, w::World)
     @ ~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:100
 [4] eat!(::Animal, ::Nothing, ::World)
     @ ~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:104

The payoff is a new species defined outside the package. Once its diet is declared, it should work without changing `eats` or `eat!`:

In [11]:
abstract type Rabbit <: AnimalSpecies end
Rabbit(id; E=2.0, ΔE=0.3, pr=0.9, pf=0.7, s=rand(instances(Sex))) = Rabbit(id, E, ΔE, pr, pf, s)


Base.show(io::IO, ::Type{Rabbit}) = print(io, "🐇")

# Extend the trait owned by the package; do not create a separate `Diet`.
Ecosystem.Diet(::Type{Rabbit}) = Ecosystem.Herbivore()

eats(Rabbit(1), Grass(2, 3, 3))   # true
eats(Wolf(1), Rabbit(2))          # true

# A species-specific rule can override the general trait rule if needed:
# Ecosystem.eats(::Animal{Wolf}, ::Animal{Rabbit}) = true

true

<div class="alert alert-block alert-info">
<b>Discussion:</b>

 Why not make `Herbivore` an `AnimalSpecies` supertype? 

 (Herbivore should not be an AnimalSpecies supertype because diet is only one property. An animal can be grouped by diet, habitat, size, etc.; traits let these groups stay independent.)
 
 Is `Ecosystem.Diet(::Type{Rabbit})` type piracy? 
 
(Ecosystem.Diet(::Type{Rabbit}) is not type piracy if you own Rabbit. You are adding behavior for your own type.)

 Can `eats(::Animal{Wolf}, ::Animal{Rabbit})` override the trait for wolves that prefer rabbits?

 (Yes. A more specific method overrides the general trait-based rule:)
</div>

## Part III: Checking the Ecosystem

Nothing above is checked by the compiler, so now bring some rules back with tests and static analysis.

<div class="alert alert-block alert-success">
<b>Exercise:</b> Add tests to `test/runtests.jl`:
</div>
1. A test set for the iteration interface of `World` (`length`, `collect`, `eltype`).
2. A test set for the `Diet` trait: wolf → sheep, sheep → grass, not sheep → wolf, and not wolf → wolf.

Run them with `]test`.


In [12]:
Pkg.test("Ecosystem")

     Testing Ecosystem
      Status `/tmp/jl_0e71ea/Project.toml`
  [3e3f0c30] Ecosystem v0.1.0 `~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4`
  [9a3f8284] Random v1.11.0
  [8dfed614] Test v1.11.0
      Status `/tmp/jl_0e71ea/Manifest.toml`
  [3e3f0c30] Ecosystem v0.1.0 `~/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4`
  [2a0f44e3] Base64 v1.11.0
  [b77e0a4c] InteractiveUtils v1.11.0
  [ac6e5ff7] JuliaSyntaxHighlighting v1.12.0
  [56ddb016] Logging v1.11.0
  [d6f4376e] Markdown v1.11.0
  [9a3f8284] Random v1.11.0
  [ea8e919c] SHA v1.0.0
  [9e88b42a] Serialization v1.11.0
  [f489334b] StyledStrings v1.11.0
  [8dfed614] Test v1.11.0
     Testing Running tests...
Test Summary:             | Pass  Total  Time
World iteration interface |    3      3  0.6s
Test Summary: | Pass  Total  Time
Diet trait    |    4      4  0.1s
Test Summary:       | Pass  Total  Time
Total animal energy |    2      2  0.2s
     Testing Ecosystem tests passed 


<div class="alert alert-block alert-success">
<b>Exercise:</b> A colleague contributed this function:
</div>

```julia
total_energy(w::World) = sum(a.enrgy for a in w if a isa Animal)
```

Use <a href="https://github.com/aviatesk/JET.jl">JET.jl</a> to find its problems without running the simulation. Fix the reported problems, then change it to `sum(a.energy for a in w)` and compare JET's report for a world of grass and sheep with one containing only grass. Finally, run `report_package(Ecosystem)`.


In [13]:
@assert isdefined(Ecosystem, :total_energy)

In [14]:
using JET

w = World([Grass(1), Sheep(2)])
@report_call Ecosystem.total_energy(w)

No errors detected


In [15]:
# JET reports a missing `energy` field only when the world is known to
# contain plants exclusively.
@report_call sum(a.energy for a in World([Grass(1), Sheep(2)]))
@report_call sum(a.energy for a in World([Grass(1), Grass(2)]))

report_package(Ecosystem)

[toplevel-info] Analyzing top-level definitions (progress: 58/58)erval: 2)
[toplevel-info] Analyzed all top-level definitions (all: 58 | analyzed: 58 | cached: 0 | took: 1.006 sec)


No errors detected


<div class="alert alert-block alert-info">
<b>Outlook:</b> Try `@report_opt world_step!(w)`. JET reports runtime dispatch. What is it, and does it matter?
</div>

In [16]:
@report_opt world_step!(w)

═════ 1 possible error found ═════
┌ world_step!(w::World{Agent}) @ Ecosystem /home/armin/Desktop/Scientific-Programming-in-Julia/interactive_notebooks/lab4/src/Ecosystem.jl:142
│ runtime dispatch detected: agent_step!(%406::Agent, w::World{Agent})::Any
└────────────────────
